# 🏦 Hosted Agent with Skills — FSI Compliance & Risk

This notebook demonstrates how to use **[Foundry Skills](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/skills?pivots=rest-api)** with a **[Hosted Agent](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents)** for Financial Services scenarios.

## Key Concepts

### 🧩 What are Skills?

[**Skills**](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/skills?pivots=rest-api) are reusable behavioral guidelines packaged as `SKILL.md` files with YAML frontmatter. They decouple domain expertise from agent code:

- **Author once, use everywhere** — Write a skill, store it centrally in Foundry, bundle into any agent
- **Version-controlled expertise** — Skills define *how* an agent should behave for a specific domain (e.g., credit risk scoring rules, compliance thresholds)
- **REST API managed** — Create, list, get, import/export, and delete skills via the Foundry Skills REST API
- **No code changes needed** — Update a skill's instructions without touching agent code; just redeploy

### 🐳 What are Hosted Agents?

[**Hosted Agents**](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents) are containerized agentic applications deployed to Azure Foundry's serverless compute:

- **Containerized** — Your agent runs as a Docker container (Python, Node.js, etc.) with full control over dependencies
- **Serverless scaling** — Azure manages the infrastructure; you pay only for what you use
- **Protocol-based** — Agents expose a standard `responses` protocol so Foundry can invoke them
- **Secure by default** — Managed identity for auth, private networking, RBAC-controlled access
- **Deploy with `azd`** — Azure Developer CLI provisions infrastructure (ACR, Foundry project, model deployments) and deploys the container

### How They Work Together

Skills are bundled inside the hosted agent's container as `SKILL.md` files. At runtime, the agent loads them via `SkillsProvider` and uses them as context to guide responses.

## What You'll Learn

1. **Author Skills** — Create `SKILL.md` files for FSI use cases
2. **Skills REST API** — Create, list, get, download, and delete skills via Foundry
3. **Bundle Skills** — Include skills in a hosted agent project
4. **Deploy** — Use `azd ai agent init` → `azd provision` → `azd deploy` to ship the agent
5. **Test Locally** — Validate skills with Agent Framework's `SkillsProvider` before deploying
6. **Test Deployed** — Invoke the hosted agent via CLI (`azd ai agent invoke`) and SDK (`get_openai_client`)

### 📚 Documentation Links

| Resource | Link |
|----------|------|
| Skills (REST API) | [learn.microsoft.com/…/skills](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/skills?pivots=rest-api) |
| Hosted Agents concepts | [learn.microsoft.com/…/hosted-agents](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents) |
| Deploy a Hosted Agent | [learn.microsoft.com/…/deploy-hosted-agent](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/deploy-hosted-agent) |
| Azure Developer CLI (`azd`) | [learn.microsoft.com/…/install-azd](https://learn.microsoft.com/azure/developer/azure-developer-cli/install-azd) |

### Architecture

```
┌──────────────────────────────────────────┐
│  Foundry Project                         │
│  ┌─────────────────┐  ┌──────────────┐  │
│  │ Skills REST API  │  │ Hosted Agent │  │
│  │ (central store)  │──│ (container)  │  │
│  └─────────────────┘  └──────────────┘  │
│         ↕ CRUD              ↑ loads      │
│  skills/                    skills/      │
│  ├── credit-risk/           ├── credit-risk/
│  │   └── SKILL.md           │   └── SKILL.md
│  └── compliance/            └── compliance/
│      └── SKILL.md               └── SKILL.md
└──────────────────────────────────────────┘
```

## 1. Setup & Authentication

In [23]:
import os
import json
import zipfile
import io
from pathlib import Path

import requests
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

# Load environment variables
load_dotenv(Path('..') / '.env')

project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
skills_endpoint = f"{project_endpoint}/skills"

# Get bearer token for Skills API
credential = AzureCliCredential()
token = credential.get_token("https://ai.azure.com/.default").token
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
    "Foundry-Features": "Skills=V1Preview",
}

print(f"🔗 Skills Endpoint: {skills_endpoint}")
print(f"🔑 Token acquired (length: {len(token)})")

🔗 Skills Endpoint: https://kd-foundry.services.ai.azure.com/api/projects/proj-kd/skills
🔑 Token acquired (length: 2859)


## 2. Author FSI Skills

Skills are `SKILL.md` files with YAML frontmatter. The `name` and `description` must be **unquoted** in YAML.

In [24]:
# Define our FSI skills as SKILL.md content

CREDIT_RISK_SKILL = """\
---
name: credit-risk-assessment
description: Assess credit risk for loan applications using financial metrics and scoring models.
---

# Credit Risk Assessment

You are a credit risk specialist. When evaluating loan applications:

## Scoring Criteria
- Credit Score (40% weight): FICO 300-850
- Debt-to-Income Ratio (30%): Monthly debt / Monthly income
- Employment Stability (15%): Years at current employer
- Collateral Coverage (15%): Collateral value / Loan amount

## Risk Categories
| Score | Category | Action |
|-------|----------|--------|
| 80-100 | Low Risk | Approve standard terms |
| 60-79 | Medium Risk | Approve with monitoring |
| 40-59 | High Risk | Require collateral/co-signer |
| 0-39 | Very High Risk | Decline or manual review |

## Rules
- Always calculate DTI ratio before making recommendations
- Flag applications with DTI > 43% for additional review
- Note that this is a simplified model for demonstration
- Comply with ECOA and Fair Housing Act requirements
"""

COMPLIANCE_SKILL = """\
---
name: transaction-compliance
description: Screen financial transactions for AML/KYC compliance and regulatory reporting requirements.
---

# Transaction Compliance Screening

You are a compliance screening specialist. When checking transactions:

## Reporting Thresholds
- Cash > $10,000: File Currency Transaction Report (CTR)
- Multiple cash transactions just under $10,000: Flag for structuring (SAR review)
- International wires > $3,000: Record and verify
- FATF grey/black list countries: Enhanced Due Diligence required

## Red Flags
1. Rapid fund movement across multiple accounts
2. Transactions inconsistent with stated business purpose
3. Use of shell companies or nominees
4. Reluctance to provide identification
5. Transactions just below reporting thresholds

## Actions
- PROCEED: No compliance concerns
- FLAG: Requires compliance officer review
- BLOCK: High-risk country or critical violation — halt transaction

## Important
- Always escalate suspicious activity to compliance officers
- This is a simplified framework for demonstration only
- Follow institution's BSA/AML program requirements
"""

print("✅ Defined 2 FSI skills:")
print("   • credit-risk-assessment")
print("   • transaction-compliance")

✅ Defined 2 FSI skills:
   • credit-risk-assessment
   • transaction-compliance


## 3. Skills REST API — CRUD Operations

### 3.1 Create Skills (JSON method)

In [25]:
def create_skill_from_json(name: str, description: str, instructions: str) -> dict:
    """Create a skill using the JSON method."""
    payload = {
        "name": name,
        "description": description,
        "instructions": instructions,
    }
    resp = requests.post(
        f"{skills_endpoint}?api-version=v1",
        headers={**headers, "Content-Type": "application/json"},
        json=payload,
    )
    if resp.status_code in (400, 409) and "already exists" in resp.text:
        # Skill already exists — fetch and return it
        print(f"   ℹ️ Skill '{name}' already exists, fetching...")
        get_resp = requests.get(
            f"{skills_endpoint}/{name}?api-version=v1",
            headers=headers,
        )
        get_resp.raise_for_status()
        return get_resp.json()
    resp.raise_for_status()
    return resp.json()


# Create credit risk skill
credit_skill = create_skill_from_json(
    name="credit-risk-assessment",
    description="Assess credit risk for loan applications using financial metrics and scoring models.",
    instructions=CREDIT_RISK_SKILL.split("---", 2)[2].strip(),  # Body after frontmatter
)
print(f"✅ Created: {credit_skill.get('name', 'unknown')}")

# Create compliance skill
compliance_skill = create_skill_from_json(
    name="transaction-compliance",
    description="Screen financial transactions for AML/KYC compliance and regulatory reporting requirements.",
    instructions=COMPLIANCE_SKILL.split("---", 2)[2].strip(),
)
print(f"✅ Created: {compliance_skill.get('name', 'unknown')}")

   ℹ️ Skill 'credit-risk-assessment' already exists, fetching...
✅ Created: credit-risk-assessment
   ℹ️ Skill 'transaction-compliance' already exists, fetching...
✅ Created: transaction-compliance


### 3.2 Import Skill from ZIP (Alternative Method)

You can also import skills from a ZIP containing `SKILL.md` at the root:

In [26]:
def import_skill_from_zip(skill_md_content: str) -> dict:
    """Import a skill by uploading a ZIP with SKILL.md at the root."""
    # Create in-memory ZIP
    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("SKILL.md", skill_md_content)
    zip_bytes = zip_buffer.getvalue()

    resp = requests.post(
        f"{skills_endpoint}:import?api-version=v1",
        headers={**headers, "Content-Type": "application/zip"},
        data=zip_bytes,
    )
    resp.raise_for_status()
    return resp.json()


# Example: Import via ZIP (uncomment to use)
# zip_result = import_skill_from_zip(CREDIT_RISK_SKILL)
# print(f"✅ Imported via ZIP: {zip_result['name']} (has_blob: {zip_result['has_blob']})")
print("💡 ZIP import available — uncomment to test (creates from binary, supports download)")

💡 ZIP import available — uncomment to test (creates from binary, supports download)


### 3.3 List Skills

In [27]:
def list_skills(limit: int = 20) -> dict:
    """List all skills in the Foundry project."""
    resp = requests.get(
        f"{skills_endpoint}?api-version=v1&limit={limit}&order=desc",
        headers=headers,
    )
    resp.raise_for_status()
    return resp.json()


skills_list = list_skills()
print(f"📋 Skills in project ({len(skills_list['data'])} total):")
for s in skills_list["data"]:
    print(f"   • {s['name']} — {s['description'][:60]}...")

📋 Skills in project (2 total):
   • transaction-compliance — Screen financial transactions for AML/KYC compliance and reg...
   • credit-risk-assessment — Assess credit risk for loan applications using financial met...


### 3.4 Get & Download a Skill

In [28]:
def get_skill(name: str) -> dict:
    """Get skill metadata by name."""
    resp = requests.get(
        f"{skills_endpoint}/{name}?api-version=v1",
        headers=headers,
    )
    resp.raise_for_status()
    return resp.json()


# Get the credit risk skill
skill_info = get_skill("credit-risk-assessment")
print(f"📄 Skill: {skill_info['name']}")
print(f"   Description: {skill_info['description']}")
print(f"   Has blob: {skill_info.get('has_blob', False)}")
print(f"   Created: {skill_info.get('created_at')}")

📄 Skill: credit-risk-assessment
   Description: Assess credit risk for loan applications using financial metrics and scoring models.
   Has blob: False
   Created: None


## 4. Generate Hosted Agent Project

The next cell creates a deployable agent project under `azure-ai-agents/my-hosted-agent/`. This includes:

- **`main.py`** — Agent entry point using Agent Framework's `ResponsesHostServer` + `SkillsProvider`
- **`requirements.txt`** — Python dependencies (`agent-framework`, `agent-framework-foundry-hosting`)
- **`Dockerfile`** — Container definition for remote build via ACR Tasks
- **`skills/`** — Bundled `SKILL.md` files for credit risk and compliance

> `azd ai agent init` (Section 5) will generate `agent.yaml`, `azure.yaml`, and `infra/` — so we only create the application code here.

In [ ]:
import textwrap

# ============================================================================
# Project root: azure-ai-agents/my-hosted-agent/
# ============================================================================
project_dir = Path("my-hosted-agent")
skills_dir = project_dir / "skills"

# Create directories
(skills_dir / "credit-risk-assessment").mkdir(parents=True, exist_ok=True)
(skills_dir / "transaction-compliance").mkdir(parents=True, exist_ok=True)

# ── SKILL.md files (UTF-8 to avoid UnicodeDecodeError) ──────────────────────
(skills_dir / "credit-risk-assessment" / "SKILL.md").write_text(
    CREDIT_RISK_SKILL, encoding="utf-8"
)
(skills_dir / "transaction-compliance" / "SKILL.md").write_text(
    COMPLIANCE_SKILL, encoding="utf-8"
)

# ── main.py (Agent Framework + ResponsesHostServer) ────────────────────────
(project_dir / "main.py").write_text(textwrap.dedent("""\
    \"\"\"Hosted Agent - Credit Risk & Compliance Skills.\"\"\"
    import os
    from pathlib import Path

    from agent_framework import Agent, SkillsProvider
    from agent_framework_foundry import FoundryChatClient
    from agent_framework_foundry_hosting import ResponsesHostServer
    from azure.identity import DefaultAzureCredential
    from dotenv import load_dotenv

    load_dotenv()

    INSTRUCTIONS = (
        "You are a financial services AI agent deployed on Azure Foundry. "
        "Use your available skills to assess credit risk and screen transactions "
        "for AML/KYC compliance. Provide clear, actionable guidance while noting "
        "regulatory disclaimers."
    )


    def main():
        credential = DefaultAzureCredential()

        client = FoundryChatClient(
            credential=credential,
            model=os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o"),
        )

        skills_provider = SkillsProvider.from_paths(
            str(Path(__file__).parent / "skills"),
        )

        agent = Agent(
            client=client,
            instructions=INSTRUCTIONS,
            context_providers=[skills_provider],
            default_options={"store": False},
        )

        server = ResponsesHostServer(agent)
        server.run()


    if __name__ == "__main__":
        main()
"""), encoding="utf-8")

# ── requirements.txt ────────────────────────────────────────────────────────
(project_dir / "requirements.txt").write_text(textwrap.dedent("""\
    agent-framework>=1.3.0
    agent-framework-foundry-hosting
    python-dotenv
    azure-identity
"""), encoding="utf-8")

# ── Dockerfile ──────────────────────────────────────────────────────────────
(project_dir / "Dockerfile").write_text(textwrap.dedent("""\
    FROM python:3.12-slim

    WORKDIR /app

    COPY requirements.txt .
    RUN pip install --no-cache-dir -r requirements.txt

    COPY . .

    EXPOSE 8080

    CMD ["python", "main.py"]
"""), encoding="utf-8")

# ── Print project tree ──────────────────────────────────────────────────────
print(f"✅ Hosted agent project created: {project_dir.absolute()}")
print("\n📁 Project structure:")
for p in sorted(project_dir.rglob("*")):
    rel = p.relative_to(project_dir)
    depth = len(rel.parts) - 1
    prefix = "    " * depth + ("└── " if p.is_file() else "├── ")
    print(f"   {prefix}{p.name}")

print(f"\n🚀 Next: cd {project_dir} && azd ai agent init")

✅ Hosted agent project created: c:\Users\kapildhanger\OneDrive - Microsoft\Microsoft_Kapil\Azure_learning\Workshops\ai_foundry_agents_Workshop\ai-immersion-day\agentic-ai-immersion-day\azure-ai-agents\my-hosted-agent

📁 Project structure:
   └── main.py
   └── requirements.txt
   ├── skills
       ├── credit-risk-assessment
           └── SKILL.md
       ├── transaction-compliance
           └── SKILL.md

🚀 Next: cd my-hosted-agent && azd ai agent init --model gpt-4o --protocol responses


## 5. Deploy the Hosted Agent to Foundry

Follow these steps to deploy the generated project as a [hosted agent on Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/agents/quickstarts/quickstart-hosted-agent?pivots=azd).

> ⚠️ **New backend:** This requires `azd ai agent` extension **v0.1.27-preview or later**. Earlier versions used Azure Container Apps — the new backend deploys directly to Foundry Agent Service.

> **Prerequisites:**
> - [Azure Developer CLI (`azd`)](https://learn.microsoft.com/azure/developer/azure-developer-cli/install-azd) v1.24.0+
> - `azd` AI agent extension: `azd ext install azure.ai.agents`
> - **Azure AI Project Manager** role at project scope (includes data plane permissions + ability to assign Azure AI User to the agent identity)
> - Hosted agents are available in [multiple regions](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents#region-availability) (East US 2, North Central US, Sweden Central, Canada Central, West US 3, and more)
> - Logged in: `azd auth login --tenant-id <your-tenant-id>`

#### Step 1: Initialize the agent project

Navigate to the generated project and run `azd ai agent init`:

```bash
cd my-hosted-agent
azd ai agent init
```

The interactive flow guides you through:

- **How do you want to initialize your agent?** → Select **"Use the code in the current directory"** (detects your existing `main.py`, `skills/`, `requirements.txt`)
- **Initialize the starter template?** → **Y** (creates `azure.yaml`, `agent.yaml`, `infra/` with ~16 Bicep template files — these are scaffolding, not deployed resources yet)
- **Enter a name for your agent** → Accept the default or type a custom name
- **Which protocols does your agent support?** → Select **responses** (use space to toggle, enter to confirm)
- **How would you like to configure a model?** → **"Select an existing model deployment from a Foundry project"** (reuses your existing Foundry project from `.env` instead of creating new infrastructure)
- **Select subscription** → Choose your Azure subscription
- **Select Foundry project** → Select the project matching your `AI_FOUNDRY_PROJECT_ENDPOINT`
- **Select model deployment** → Select your existing **gpt-4o** deployment
- **Container size** → Accept defaults (0.25 vCPU / 0.5 GiB) or choose larger

> **Why so many infra files?** The `infra/` folder contains Bicep templates for *both* scenarios — creating new resources and using existing ones (notice `existing-ai-project.bicep`). When you select an existing Foundry project, the init step provisions only the ACR and monitoring resources; it does **not** duplicate your Foundry project or model deployments.

After completion:
```
Added your agent as a service entry named '<agent-name>' under the file azure.yaml.
  +  agent.yaml

You can customize environment variables and other settings in the agent.yaml.
Next steps: Run azd deploy <agent-name> to deploy your agent to Microsoft Foundry.
```

> **Note the agent name** — you'll need it for `azd deploy <agent-name>` in Step 3.

#### What `azd ai agent init` creates

| File | Purpose |
|------|---------|
| `azure.yaml` | azd project definition with your agent as a service |
| `agent.yaml` | Agent container definition (protocols, env vars) |
| `infra/main.bicep` | Bicep template for Foundry project, ACR, model deployments |
| `.azure/<env>/.env` | Environment config (subscription, location, model) |

#### Step 2: Provision infrastructure

```bash
azd provision
```

This creates the Azure Container Registry (ACR) and monitoring resources needed for hosting. When using an existing Foundry project, it does **not** create a new project or model deployments — those are reused.

> You need **Contributor** access on your Azure subscription for provisioning.

#### Step 3: Deploy to Foundry Agent Service

```bash
azd deploy <agent-name>
```

Replace `<agent-name>` with the service name from `azd ai agent init` output (e.g., `risk-assement-hosted-agent`).

The container is built remotely via ACR Tasks — Docker Desktop is **not** required (as long as `azd provision` created the ACR). This also assigns RBAC roles to the agent's identity (requires **Owner** or **User Access Administrator**).

> **Tip:** You can also run `azd up` to combine provision + deploy in one command.

#### Step 4: Verify and test the deployed agent

```bash
# Check agent status
azd ai agent show --output table

# Test credit risk assessment (Responses protocol — plain string)
azd ai agent invoke "Assess this mortgage: credit score 680, DTI 38%, LTV 90%, 4 years employment"

# Test compliance screening — sanctioned country
azd ai agent invoke "Screen this transaction: $25,000 wire transfer to a company in Iran"

# Test structuring detection
azd ai agent invoke "A customer made 3 cash deposits in one week: $9,500, $9,800, and $9,200. Should we flag this?"

# Test high-risk applicant
azd ai agent invoke "Evaluate: credit score 480, DTI 55%, no collateral, 2 months at job, requesting $100,000"

# Test combined scenario (both skills)
azd ai agent invoke "Customer with credit score 710 applying for $200K loan also wants to wire $15K to Turkey. Assess both."

# View live logs
azd ai agent monitor --follow

# View last 20 lines of console logs
azd ai agent monitor --tail 20
```

You can also test in the [Foundry playground](https://ai.azure.com/) — navigate to **Build → Agents → your agent → Open in playground**.

#### Updating Skills

When you update a skill, simply redeploy — no code changes needed:

```bash
# Edit skills/credit-risk-assessment/SKILL.md
azd deploy <agent-name>
```

## 6. Test Skills Locally

Test the skills locally using the Agent Framework's `SkillsProvider` — this validates that bundled skills load correctly and produce expected responses. You can run this before *or* after deploying:

In [30]:
from agent_framework import SkillsProvider
from agent_framework_foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential
import os


async def test_hosted_agent_skills_locally():
    """Test the bundled skills locally before deploying."""
    # Load skills from the same directory structure used in the hosted agent
    skills_provider = SkillsProvider.from_paths(
        skills_dir.absolute(),
    )

    async with AzureCliCredential() as credential:
        client = FoundryChatClient(credential=credential)
        agent = client.as_agent(
            name="FSIHostedAgent",
            instructions=(
                "You are a financial services AI agent deployed on Azure Foundry. "
                "Use your available skills to assess credit risk and screen transactions. "
                "Provide clear, actionable guidance while noting regulatory disclaimers."
            ),
            context_providers=[skills_provider],
        )

        print("=== 🏦 FSI Hosted Agent — Local Skills Test ===")
        print("=" * 55)

        # ── Test 1: Credit risk — standard mortgage ─────────────────────
        q1 = (
            "Evaluate this mortgage application: "
            "Credit score 680, annual income $120,000, monthly debt $3,200, "
            "loan amount $450,000, 4 years employment, property value $500,000."
        )
        print(f"\n📋 Test 1 (Credit Risk — Medium): {q1[:80]}...")
        r1 = await agent.run(q1)
        print(f"🤖 Response:\n{r1.text[:500]}...\n")

        # ── Test 2: Credit risk — high risk applicant ───────────────────
        q2 = (
            "Assess this personal loan: credit score 520, DTI 48%, "
            "no collateral, 6 months at current job, requesting $50,000."
        )
        print(f"📋 Test 2 (Credit Risk — High Risk): {q2[:80]}...")
        r2 = await agent.run(q2)
        print(f"🤖 Response:\n{r2.text[:500]}...\n")

        # ── Test 3: Credit risk — low risk applicant ────────────────────
        q3 = (
            "Evaluate: credit score 790, DTI 22%, 15 years employment, "
            "LTV 60%, requesting $300,000 mortgage."
        )
        print(f"📋 Test 3 (Credit Risk — Low Risk): {q3[:80]}...")
        r3 = await agent.run(q3)
        print(f"🤖 Response:\n{r3.text[:500]}...\n")

        # ── Test 4: Compliance — sanctioned country wire ────────────────
        q4 = "Screen this: $25,000 wire transfer to a company in Iran."
        print(f"📋 Test 4 (Compliance — BLOCK): {q4}")
        r4 = await agent.run(q4)
        print(f"🤖 Response:\n{r4.text[:500]}...\n")

        # ── Test 5: Compliance — structuring detection ──────────────────
        q5 = (
            "A customer made 3 cash deposits in one week: $9,500, $9,800, "
            "and $9,200. All to the same account. Should we flag this?"
        )
        print(f"📋 Test 5 (Compliance — Structuring): {q5[:80]}...")
        r5 = await agent.run(q5)
        print(f"🤖 Response:\n{r5.text[:500]}...\n")

        # ── Test 6: Compliance — legitimate transaction ─────────────────
        q6 = (
            "Screen: domestic wire of $5,000 from a verified business account "
            "to a known supplier for inventory purchase."
        )
        print(f"📋 Test 6 (Compliance — PROCEED): {q6[:80]}...")
        r6 = await agent.run(q6)
        print(f"🤖 Response:\n{r6.text[:500]}...\n")

        print("=" * 55)
        print("✅ All 6 local tests completed")


await test_hosted_agent_skills_locally()

<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


=== 🏦 FSI Hosted Agent — Local Skills Test ===

📋 Test 1 (Credit Risk — Medium): Evaluate this mortgage application: Credit score 680, annual income $120,000, mo...
🤖 Response:
Let's evaluate the mortgage application using the credit risk assessment criteria:

### Applicant Information:
- **Credit Score**: 680
- **Annual Income**: $120,000
- **Monthly Debt**: $3,200
- **Loan Amount**: $450,000
- **Years at Current Employer**: 4
- **Property Value**: $500,000

### Assessment Criteria:

1. **Credit Score (40% Weight)**
   - Scaled score: \( \frac{680 - 300}{850 - 300} \times 40 = 27.6 \)

2. **Debt-to-Income (DTI) Ratio (30% Weight)**
   - DTI Ratio: \( \frac{3,200}{10,0...

📋 Test 2 (Credit Risk — High Risk): Assess this personal loan: credit score 520, DTI 48%, no collateral, 6 months at...
🤖 Response:
Based on the credit risk assessment criteria, here's the evaluation for the given personal loan application:

1. **Credit Score (40% weight)**
   - Applicant's Score: 520
   - Risk: High

## 7. Test the Deployed Agent (SDK)

After deploying, you can invoke the hosted agent programmatically using `AIProjectClient.get_openai_client()`. This calls the deployed container's `/responses` endpoint — the same thing `azd ai agent invoke` does, but from Python:

In [32]:
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

# ── Connect to the Foundry project ──────────────────────────────────────────
# allow_preview=True is required to use get_openai_client with agent_name
project_client = AIProjectClient(
    endpoint=os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
    allow_preview=True,
)

# ── Get an OpenAI client bound to the deployed agent ────────────────────────
# Replace with your agent name from `azd ai agent init`
AGENT_NAME = "risk-assement-hosted-agent"
openai_client = project_client.get_openai_client(agent_name=AGENT_NAME)

print(f"🔗 Connected to deployed agent: {AGENT_NAME}")
print("=" * 55)

# ── Test 1: Credit risk assessment ──────────────────────────────────────────
print("\n📋 Test 1: Credit Risk — Medium Risk Mortgage")
response = openai_client.responses.create(
    input=(
        "Evaluate this mortgage: credit score 680, annual income $120,000, "
        "monthly debt $3,200, loan amount $450,000, 4 years employment, "
        "property value $500,000."
    ),
)
print(f"🤖 {response.output_text[:600]}...\n")

# ── Test 2: High-risk applicant ─────────────────────────────────────────────
print("📋 Test 2: Credit Risk — Very High Risk")
response = openai_client.responses.create(
    input=(
        "Assess: credit score 480, DTI 55%, 2 months employment, "
        "no collateral, requesting $100,000 unsecured loan."
    ),
)
print(f"🤖 {response.output_text[:600]}...\n")

# ── Test 3: Compliance — structuring detection ──────────────────────────────
print("📋 Test 3: Compliance — Structuring Pattern")
response = openai_client.responses.create(
    input=(
        "Customer made 4 cash deposits this week: $9,500, $9,800, $9,200, "
        "and $9,900. All to the same account. Assess for compliance."
    ),
)
print(f"🤖 {response.output_text[:600]}...\n")

# ── Test 4: Compliance — sanctioned country ─────────────────────────────────
print("📋 Test 4: Compliance — Sanctioned Country")
response = openai_client.responses.create(
    input="Screen: $50,000 wire transfer to a company in North Korea.",
)
print(f"🤖 {response.output_text[:600]}...\n")

# ── Test 5: Compliance — legitimate transaction ─────────────────────────────
print("📋 Test 5: Compliance — Clean Transaction")
response = openai_client.responses.create(
    input=(
        "Screen: $3,500 domestic ACH payment from a verified payroll account "
        "to an employee's checking account for regular salary."
    ),
)
print(f"🤖 {response.output_text[:600]}...\n")

# ── Test 6: Combined scenario ──────────────────────────────────────────────
print("📋 Test 6: Combined — Loan + Compliance")
response = openai_client.responses.create(
    input=(
        "A business customer with credit score 710 and DTI 35% is applying for a "
        "$200,000 business loan. They also want to wire $15,000 to a supplier in Turkey. "
        "Assess both the credit risk and compliance aspects."
    ),
)
print(f"🤖 {response.output_text[:800]}...\n")

print("=" * 55)
print("✅ All 6 deployed agent tests completed")

🔗 Connected to deployed agent: risk-assement-hosted-agent

📋 Test 1: Credit Risk — Medium Risk Mortgage
🤖 Here’s an evaluation of the mortgage application based on the provided model.

Inputs:
- Credit score: 680  
- Annual income: $120,000 → Monthly income ≈ $10,000  
- Monthly debt (existing obligations): $3,200  
- Loan amount: $450,000  
- Property value: $500,000  
- Employment: 4 years

---

### 1. Key Ratios & Metrics

**Debt-to-Income (DTI)**  
- DTI = Monthly debt / Monthly income  
- DTI = 3,200 / 10,000 = 32%

This is below the 43% flag threshold.

**Loan-to-Value (LTV)**  
- LTV = Loan amount / Property value  
- LTV = 450,000 / 500,000 = 90%  
- Collateral coverage = Property value / ...

📋 Test 2: Credit Risk — Very High Risk
🤖 Assessment based on the provided profile:

- Credit score: 480 (very weak)
- DTI: 55% (well above typical acceptable thresholds; flagged for high risk)
- Employment: 2 months (no demonstrated stability)
- Collateral: None (unsecured request)
- Loan

## 8. Cleanup — Delete Skills

In [ ]:
def delete_skill(name: str) -> dict:
    """Delete a skill from Foundry."""
    resp = requests.delete(
        f"{skills_endpoint}/{name}?api-version=v1",
        headers=headers,
    )
    resp.raise_for_status()
    return resp.json()


# ⚠️ Uncomment to delete skills from Foundry
# result = delete_skill("credit-risk-assessment")
# print(f"🗑️ Deleted: credit-risk-assessment — {result}")
# result = delete_skill("transaction-compliance")
# print(f"🗑️ Deleted: transaction-compliance — {result}")

print("💡 Uncomment above to delete skills from Foundry")

## 9. Cleanup — Delete the Hosted Agent

To prevent charges, clean up the hosted agent and its provisioned infrastructure when finished. Agent compute is deprovisioned after 15 minutes of inactivity, so there's no cost when an agent isn't serving requests.

> ⚠️ **Warning:** Deleting an agent removes **all its versions** and terminates active sessions. This action can't be undone.

There are three ways to clean up:

#### Option A: Azure Developer CLI (`azd down`)
Tears down all provisioned infrastructure (ACR, monitoring, agent):
```bash
cd my-hosted-agent
azd down
```

#### Option B: SDK — delete a version or the entire agent
```python
project.agents.delete_version(agent_name="my-agent", agent_version=agent.version)  # single version
project.agents.delete(agent_name="my-agent")  # entire agent + all versions
```

#### Option C: REST API
```bash
# Delete a single version
curl -X DELETE "$BASE_URL/agents/my-agent/versions/1?api-version=$API_VERSION" \
  -H "Authorization: Bearer $TOKEN"

# Or delete the entire agent
curl -X DELETE "$BASE_URL/agents/my-agent?api-version=$API_VERSION" \
  -H "Authorization: Bearer $TOKEN"
```

See: [Clean up resources](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/deploy-hosted-agent#clean-up-resources)

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

AGENT_NAME = "risk-assement-hosted-agent"  # your agent name from azd ai agent init

project_client = AIProjectClient(
    endpoint=os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
    allow_preview=True,
)

# ⚠️ Uncomment ONE of the options below to clean up the hosted agent

# ── Option 1: Delete a specific version ─────────────────────────────────────
# AGENT_VERSION = 2  # replace with your version number
# project_client.agents.delete_version(agent_name=AGENT_NAME, agent_version=AGENT_VERSION)
# print(f"🗑️ Deleted version {AGENT_VERSION} of agent '{AGENT_NAME}'")

# ── Option 2: Delete the entire agent and ALL versions ──────────────────────
# project_client.agents.delete(agent_name=AGENT_NAME)
# print(f"🗑️ Deleted agent '{AGENT_NAME}' and all its versions")

print(f"💡 Uncomment above to delete hosted agent '{AGENT_NAME}' from Foundry")
print("   Or run 'azd down' from the my-hosted-agent/ directory to tear down all infrastructure")

## Summary

| Concept | Details |
|---------|----------|
| **[Skills](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/skills?pivots=rest-api)** | Reusable behavioral guidelines (`SKILL.md`) managed via REST API |
| **[Hosted Agents](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents)** | Containerized agents deployed to Foundry Agent Service |
| **Agent Framework** | `Agent` + `SkillsProvider` + `ResponsesHostServer` for the Responses protocol |
| **Skills REST API** | `{endpoint}/skills` — CRUD operations for central skill storage |
| **Create (JSON)** | `POST /skills` with name, description, instructions |
| **Import (ZIP)** | `POST /skills:import` with ZIP containing `SKILL.md` at root |
| **Download** | `GET /skills/{name}:download` — only for ZIP-imported skills |
| **Deploy** | `azd ai agent init` → `azd provision` → `azd deploy <agent-name>` |
| **Test (CLI)** | `azd ai agent invoke "..."` — sends to deployed container |
| **Test (SDK)** | `project_client.get_openai_client(agent_name=...)` → `responses.create()` |
| **Cleanup** | `azd down` or `project.agents.delete(agent_name=...)` |
| **Auth** | Bearer token with scope `https://ai.azure.com/.default` |
| **RBAC** | Azure AI Project Manager at project scope |

### Deployment Workflow

```
1. Author SKILL.md files
2. Import to Foundry (REST API) for central management
3. Bundle skills in agent project's skills/ directory
4. azd ai agent init → configure model, protocol, container size
5. azd provision (creates ACR + monitoring)
6. azd deploy <agent-name> (build + deploy container)
7. azd ai agent invoke "..." (test deployed agent via CLI)
8. project_client.get_openai_client() (test via SDK)
9. azd down / project.agents.delete() (cleanup)
```

### Next Steps
- **Quickstart**: [Deploy your first hosted agent](https://learn.microsoft.com/en-us/azure/foundry/agents/quickstarts/quickstart-hosted-agent?pivots=azd)
- **Skills documentation**: [learn.microsoft.com/…/skills](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/skills?pivots=rest-api)
- **Hosted Agents concepts**: [learn.microsoft.com/…/hosted-agents](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents)
- **Manage hosted agents**: [learn.microsoft.com/…/manage-hosted-agent](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-agent)
- **Agent Framework Skills**: See `../agent-framework/skills/` for file-based + code-defined skills
- **Curate Toolboxes**: See notebook 7 (MCP Tools) for intent-based toolbox integration